In [ ]:
import subprocess, os, shutil

REPO_URL = "https://github.com/safety-research/legibility.git"
REPO_DIR = "/workspace/18-4-2026"
EXP_DIR = os.path.join(REPO_DIR, "experiments", "2026", "15-4-2026")

# Clone or pull latest (fetch + reset to ensure we have the newest commit)
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

# Install git-lfs if not available, then pull LFS files
if shutil.which("git-lfs") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=False)
    subprocess.run(["git", "lfs", "install"], check=False)
subprocess.run(["git", "-C", REPO_DIR, "lfs", "pull"], check=False)

# Install dependencies
req_path = os.path.join(EXP_DIR, "requirements.txt")
if os.path.exists(req_path):
    subprocess.run(["pip", "install", "-q", "-r", req_path], check=True)
else:
    print(f"WARNING: {req_path} not found, skipping pip install")

# Set working directory so Path.cwd().parent resolves to experiment root
os.chdir(os.path.join(EXP_DIR, "notebooks"))

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from phase2_utils import PHASE2_RESULTS_DIR

_output_path = PHASE2_RESULTS_DIR / 'reader_r5_analysis_results.json'
if _output_path.exists():
    import json as _json
    with open(_output_path) as _f:
        _saved = _json.load(_f)
    print(f"CACHED: {_output_path} exists.")
    for key in _saved:
        n_layers = len(_saved[key]) if isinstance(_saved[key], dict) else 'N/A'
        print(f"  {key}: {n_layers} layers")
    print("Delete this file and re-run to recompute.")
else:
    print("No cached R5 analysis results -- will compute from scratch.")

# NB07b: Reader R5 Activation Analysis (Experiment D)

**CPU notebook** (~15 min). Analyzes R5 activations to understand what happens
inside the Google-lineage reader when it succeeds vs fails at C2 crossfill.

Mirrors NB07 (R2 analysis) but uses R5 (Gemma-4-31B-IT) activations.
Together with R2, this tests whether reader-side patterns generalize across
model families (Meta vs Google).

**Requires:** NB02b outputs (`activations/R5_last_token/`, `activations/R5_cot_boundary/`)
**Optional:** NB10b outputs (`distributional_shift_scores.json` with R5 entries) for perplexity covariate

In [ ]:
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path.cwd().parent))
from phase2_utils import (
    load_activations, load_distributional_shift_scores, train_binary_probe,
    permutation_test, plot_layer_probe_curve, bootstrap_ci_metric,
    ACTIVATIONS_DIR, PHASE2_RESULTS_DIR,
)

PHASE2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load R5 activations and metadata
r5_last = load_activations(ACTIVATIONS_DIR / "R5_last_token")
r5_boundary = load_activations(ACTIVATIONS_DIR / "R5_cot_boundary")

with open(ACTIVATIONS_DIR / "R5_last_token" / "metadata.json") as f:
    r5_meta = json.load(f)

r5_labels = np.array(r5_meta['labels'])  # 1=C2 success, 0=C2 failure
r5_sample_meta = r5_meta['sample_metadata']

print(f"R5 activations: {len(r5_labels)} samples, layers={sorted(r5_last.keys())}")
print(f"C2 success rate: {r5_labels.mean():.1%}")

# Verify activation shapes match R5 architecture
sample_layer = list(r5_last.keys())[0]
print(f"Activation shape at layer {sample_layer}: {r5_last[sample_layer].shape}")
assert r5_last[sample_layer].shape[1] == 5376, f"Expected hidden_dim=5376, got {r5_last[sample_layer].shape[1]}"

# Check by legibility class
for label in ['ANSWER_LEAKED', 'REASONING_LEGIBLE', 'ILLEGIBLE']:
    mask = np.array([m['label'] == label for m in r5_sample_meta])
    if mask.sum() > 0:
        print(f"  {label}: n={mask.sum()}, C2 success={r5_labels[mask].mean():.1%}")

In [ ]:
# Probe 1: Predict C2 success from last-token activations
# IMPORTANT: Exclude ANSWER_LEAKED samples -- these trivially succeed at C2
# and would inflate probe AUROC. Only use REASONING_LEGIBLE + ILLEGIBLE.
print("=== Probe: Predict C2 Success (excluding ANSWER_LEAKED) ===")

non_leaked_mask = np.array([
    m['label'] in ('REASONING_LEGIBLE', 'ILLEGIBLE')
    for m in r5_sample_meta
])
non_leaked_labels = r5_labels[non_leaked_mask]

print(f"Non-leaked samples: {non_leaked_mask.sum()} "
      f"(C2 success={non_leaked_labels.sum()}, failure={(1-non_leaked_labels).sum()})")

n_min_class = min(non_leaked_labels.sum(), (1 - non_leaked_labels).sum())
if n_min_class < 5:
    print(f"WARNING: Min class has only {n_min_class} samples -- probe may be unreliable")

c2_success_results = {}
for layer_idx in sorted(r5_last.keys()):
    features = r5_last[layer_idx][non_leaked_mask]
    if n_min_class < 5:
        print(f"  Layer {layer_idx}: skipping (min class={n_min_class})")
        continue

    result = train_binary_probe(features, non_leaked_labels, n_splits=min(5, n_min_class))
    c2_success_results[layer_idx] = result
    print(f"  Layer {layer_idx:3d}: AUROC={result['auroc']:.3f} CI={result['auroc_ci']}")

if c2_success_results:
    fig, ax = plot_layer_probe_curve(
        c2_success_results,
        title='Exp D: C2 Success Probe (R5, excl. ANSWER_LEAKED)',
        save_path=str(PHASE2_RESULTS_DIR / 'd_c2_success_probe_R5.png'),
    )
    plt.show()

In [ ]:
# Probe 2: Predict C2 success from CoT boundary activations
# Exclude ANSWER_LEAKED (same filter as Probe 1)
print("=== Probe: Predict C2 Success (CoT boundary, excl. ANSWER_LEAKED) ===")

boundary_results = {}
for layer_idx in sorted(r5_boundary.keys()):
    features = r5_boundary[layer_idx][non_leaked_mask]
    if n_min_class < 5:
        continue

    result = train_binary_probe(features, non_leaked_labels, n_splits=min(5, n_min_class))
    boundary_results[layer_idx] = result
    print(f"  Layer {layer_idx:3d}: AUROC={result['auroc']:.3f} CI={result['auroc_ci']}")

In [ ]:
# Probe 3: Predict legibility class from R5 activations
# (excluding ANSWER_LEAKED)
print("=== Probe: Predict Legibility from R5 Activations ===")

leg_mask = np.array([
    m['label'] in ('REASONING_LEGIBLE', 'ILLEGIBLE')
    for m in r5_sample_meta
])
leg_labels = np.array([
    1 if m['label'] == 'REASONING_LEGIBLE' else 0
    for m in r5_sample_meta
])

print(f"Non-leaked samples: {leg_mask.sum()} "
      f"(legible={leg_labels[leg_mask].sum()}, illegible={(1-leg_labels[leg_mask]).sum()})")

if leg_mask.sum() >= 20:
    legibility_from_reader_results = {}
    for layer_idx in sorted(r5_last.keys()):
        features = r5_last[layer_idx][leg_mask]
        labels = leg_labels[leg_mask]

        result = train_binary_probe(features, labels, n_splits=5)
        legibility_from_reader_results[layer_idx] = result
        print(f"  Layer {layer_idx:3d}: AUROC={result['auroc']:.3f} CI={result['auroc_ci']}")
else:
    print("Insufficient non-leaked samples for this analysis")
    legibility_from_reader_results = {}

In [ ]:
# R5 perplexity covariate analysis
ds_scores = load_distributional_shift_scores()

# Get R5 perplexity for each sample
r5_perplexity = []
for m in r5_sample_meta:
    key = (m['sample_id'], m['generator_id'], m['epoch'], 'R5')
    entry = ds_scores.get(key)
    r5_perplexity.append(entry['reader_perplexity'] if entry else np.nan)
r5_perplexity = np.array(r5_perplexity, dtype=float)

valid_p = np.isfinite(r5_perplexity)
print(f"R5 perplexity available: {valid_p.sum()}/{len(r5_perplexity)}")

if valid_p.sum() >= 20 and c2_success_results:
    # Compare C2 success probe with vs without R5 perplexity
    best_layer = max(c2_success_results, key=lambda k: c2_success_results[k]['auroc'])
    # Apply both non_leaked and valid_perplexity masks
    combined_mask = non_leaked_mask & valid_p
    features_base = r5_last[best_layer][combined_mask]
    labels_p = r5_labels[combined_mask]
    perplexity_p = r5_perplexity[combined_mask]

    print(f"\nPerplexity covariate analysis at best layer {best_layer}:")
    print(f"  Samples with both labels and perplexity: {combined_mask.sum()}")

    # Without R5 perplexity
    r_no_p = train_binary_probe(features_base, labels_p)
    print(f"  Without R5 perplexity: AUROC={r_no_p['auroc']:.3f} CI={r_no_p['auroc_ci']}")

    # With R5 perplexity
    features_with_p = np.column_stack([features_base, perplexity_p.reshape(-1, 1)])
    r_with_p = train_binary_probe(features_with_p, labels_p)
    print(f"  With R5 perplexity:    AUROC={r_with_p['auroc']:.3f} CI={r_with_p['auroc_ci']}")

    # R5 perplexity only
    r_p_only = train_binary_probe(perplexity_p.reshape(-1, 1), labels_p)
    print(f"  R5 perplexity only:    AUROC={r_p_only['auroc']:.3f} CI={r_p_only['auroc_ci']}")
elif valid_p.sum() < 20:
    print("Insufficient R5 perplexity data -- run NB10b first.")
    print("Skipping perplexity covariate analysis.")

In [ ]:
# Save results
def clean_results(d):
    return {int(k): {kk: vv for kk, vv in v.items() if kk not in ('probe_model', 'scaler', 'pipeline')}
            for k, v in d.items()}

output = {
    'c2_success_probe': clean_results(c2_success_results),
    'boundary_probe': clean_results(boundary_results),
    'legibility_from_reader': clean_results(legibility_from_reader_results),
}
output_path = PHASE2_RESULTS_DIR / 'reader_r5_analysis_results.json'
with open(output_path, 'w') as f:
    json.dump(output, f, indent=2, default=str)
print(f"Saved to {output_path}")

# Print summary
print("\n--- R5 Analysis Summary ---")
for probe_name, results in [('C2 Success (last-token)', c2_success_results),
                             ('C2 Success (boundary)', boundary_results),
                             ('Legibility', legibility_from_reader_results)]:
    if results:
        best_layer = max(results, key=lambda k: results[k]['auroc'])
        best_auroc = results[best_layer]['auroc']
        print(f"  {probe_name}: best layer={best_layer}, AUROC={best_auroc:.3f}")